# 02 · 資料清理:髒資料變乾淨

> 「資料科學家 80% 的時間在清資料,剩下 20% 在抱怨要清資料。」

這句玩笑話很真實。真實資料總是**有缺失、型別不對、有重複、有離群值**。模型再強,餵髒資料進去也是垃圾進垃圾出。這課把 Titanic 清乾淨。

## 1. 載入資料

In [ ]:
%pip install -q -U seaborn scikit-learn

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")    # 真實公開資料集:鐵達尼號乘客
print("資料形狀:", df.shape)        # (891, 15) = 891 位乘客 × 15 個欄位
df.head()

## 2. 找出缺失值

第一步永遠是:每一欄缺幾個?

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])
# deck 缺超多、age 缺一些、embarked/embark_town 缺幾個——各有對策。

## 3. 對症下藥:刪 vs 補

- **缺太多**(deck 缺近 7 成):整欄資訊太少,直接刪欄。
- **缺一些、且重要**(age):用中位數補(比平均更抗離群)。
- **缺很少**(embarked):用最常見值(眾數)補。

In [ ]:
clean = df.copy()
clean = clean.drop(columns=["deck"])                       # 缺太多,刪欄
clean["age"] = clean["age"].fillna(clean["age"].median())  # 中位數補
clean["embarked"] = clean["embarked"].fillna(clean["embarked"].mode()[0])
clean["embark_town"] = clean["embark_town"].fillna(clean["embark_town"].mode()[0])
print("還有缺值嗎?", clean.isna().sum().sum(), "個")

## 4. 重複值與離群值

重複列直接刪;離群值先用分位數看,別急著刪(它可能是真實的極端)。

In [ ]:
print("重複列:", clean.duplicated().sum(), "筆")
clean = clean.drop_duplicates()

# 票價 fare 的離群:看 99 百分位 vs 最大值
q99 = clean["fare"].quantile(0.99)
print(f"fare 99% 分位 = {q99:.0f},最大值 = {clean['fare'].max():.0f}")
print("→ 有幾個超貴的頭等艙票價。先保留,記住它存在即可。")

## 小結

- 清理四件事:**缺失值、型別、重複、離群**。
- 缺失值要**對症下藥**:缺太多刪欄、重要的補中位數、少量的補眾數。
- 離群值別反射性刪掉——先理解它是錯誤還是真實極端。

下一課:資料乾淨了,開始**探索**——找出哪些因素跟生還有關。